# Playbook 1 · The as-of join

**Stage:** one SQL query producing one row per weather zone and target hour,
carrying the forecast vintage that was actually available at `T − 24h`, the
reported actual, and the error.


> **Playbook, not walkthrough.** [`exploration.ipynb`](exploration.ipynb) is the
> narrative for a reviewer: what was built and why. These five are operational —
> one per assignment stage, each answering *what does this stage guarantee* and
> *what would it take to run it in production*. They overlap deliberately on
> evidence and not at all on purpose.


## What this stage must guarantee

> Nothing used to produce a forecast for target hour `T` may have become
> available after `T − 24h`.

A violation of this rule **throws no error**. It produces a better-looking
number. That asymmetry is the entire reason the readiness gate re-checks the
predicate with its own independent query instead of trusting this one.

**Sign convention: `error = forecast − actual`.** A positive error is an
over-forecast.

## The three clocks

The most common way this goes quietly wrong is collapsing the cutoff into one
global "as of yesterday" timestamp. It is **per target hour** — the forecast for
03:00 and the forecast for 22:00 have deadlines 19 hours apart.

| Selected | Clock | Predicate |
| --- | --- | --- |
| ERCOT forecast | decision time | `publication_ts <= target_ts − 24h` |
| Seasonal-naive input | decision time | `publication_ts <= target_ts − 24h` |
| Reported actual | evaluation time | `publication_ts <= processing_ts` |

The actual is **truth, not an input** — it is allowed to arrive after the target
hour, because that is the point of evaluating later. The naive input is a
**model input**, so it clears the same 24-hour bar as the forecast. Getting that
backwards is the leak worth probing for.

In [1]:
from __future__ import annotations

import datetime as dt
import tempfile
import textwrap
from pathlib import Path

from forecast_spine import coverage, fixtures, gates, normalize, pipeline, seasonal_naive

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw"
SQL = REPO / "sql" / "asof_join.sql"

# The assignment window. Actuals for operating day D publish on D+1, so the
# processing date is one day past the last target day.
PROCESSING_DATE = dt.date(2026, 3, 24)
WINDOW_START, WINDOW_END = dt.date(2026, 2, 22), dt.date(2026, 3, 23)
PUBLICATION_START = dt.date(2026, 2, 21)

LIVE = any(RAW.glob("load_forecast/*_csv.zip"))
raw_root = RAW if LIVE else fixtures.build("pass", Path(tempfile.mkdtemp()))
if not LIVE:
    print("LIVE VINTAGES NOT FOUND -- using synthetic fixtures.\n"
          "Structure is preserved; scale and revision behaviour are not.\n")


def build():
    """Build a throwaway warehouse. Never touches data/warehouse/.

    A scratch database keeps this notebook runnable while something else holds
    the committed one -- DuckDB is single-writer, and a SQL client with an open
    connection is enough to block it.
    """
    if LIVE:
        context = pipeline.build_context(
            PROCESSING_DATE, raw_root, window_start=WINDOW_START, window_end=WINDOW_END
        )
    else:
        context = pipeline.build_context(
            fixtures.processing_date_for("pass"), raw_root, window_days=1
        )
    con = pipeline.connect(Path(tempfile.mkdtemp()) / "playbook.duckdb")
    pipeline.load(con, context)
    pipeline.build_evaluation_dataset(con, context, SQL)
    return con, context

## Run it — one row, fully annotated

In [2]:
con, context = build()
con.execute("SET TimeZone='UTC'")

zone, day, he = ("COAST", dt.date(2026, 3, 10), 18) if LIVE else (
    con.execute("SELECT weather_zone, operating_date, hour_ending FROM evaluation_dataset LIMIT 1").fetchone()
)

row = con.execute(
    """
    SELECT target_ts_utc, cutoff_ts_utc, ercot_publication_ts_utc, ercot_vintage_age_hours,
           ercot_forecast_mw, actual_mw, actual_publication_ts_utc,
           naive_source_ts_utc, naive_source_publication_ts_utc, naive_forecast_mw,
           ercot_error_mw, naive_error_mw
    FROM evaluation_dataset
    WHERE weather_zone = ? AND operating_date = ? AND hour_ending = ?
    """,
    [zone, day, he],
).df().iloc[0]

for field, value in row.items():
    print(f"  {field:34s} {value}")

ever, eligible = con.execute(
    """
    SELECT count(*),
           count(*) FILTER (WHERE publication_ts_utc <= target_ts_utc - INTERVAL 24 HOUR)
    FROM forecast_vintage
    WHERE weather_zone = ? AND is_ercot_model_in_use
      AND target_ts_utc = (SELECT target_ts_utc FROM evaluation_dataset
                           WHERE weather_zone = ? AND operating_date = ? AND hour_ending = ?)
    """,
    [zone, zone, day, he],
).fetchone()
print(f"\n  forecasts ever published for this hour : {ever}")
print(f"  ...eligible at the cutoff              : {eligible}")
print(f"  the query takes the LAST of the {eligible}, not the BEST of the {ever}.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  target_ts_utc                      2026-03-10 22:00:00+00:00
  cutoff_ts_utc                      2026-03-09 22:00:00+00:00
  ercot_publication_ts_utc           2026-03-09 21:30:00+00:00
  ercot_vintage_age_hours            0.5
  ercot_forecast_mw                  15653.3711
  actual_mw                          16315.5
  actual_publication_ts_utc          2026-03-11 10:50:00+00:00
  naive_source_ts_utc                2026-03-03 23:00:00+00:00
  naive_source_publication_ts_utc    2026-03-04 11:50:00+00:00
  naive_forecast_mw                  15402.91
  ercot_error_mw                     -662.1288999999997
  naive_error_mw                     -912.5900000000001

  forecasts ever published for this hour : 186
  ...eligible at the cutoff              : 155
  the query takes the LAST of the 155, not the BEST of the 186.


Two things in that row are worth pausing on.

`actual_publication_ts_utc` is **later than the target hour**. Correct and
deliberate — the actual is evaluation truth, bounded by `processing_ts` rather
than the 24-hour cutoff.

And on a week spanning 8 March, `naive_source_ts_utc` to `target_ts_utc` is
**167 hours, not 168**. Joining on `target_ts - INTERVAL 168 HOUR` would silently
select the wrong hour twice a year. The join is on
`(operating_date − 7, hour_ending, dst_flag)` instead.

## What a relaxed predicate is worth

In [3]:
asof, hindsight = con.execute(
    """
    WITH latest AS (
        SELECT weather_zone, target_ts_utc, forecast_mw,
               row_number() OVER (PARTITION BY weather_zone, target_ts_utc
                                  ORDER BY publication_ts_utc DESC) AS rn
        FROM forecast_vintage
        WHERE is_ercot_model_in_use AND weather_zone <> 'SYSTEM_TOTAL'
    )
    SELECT 100 * sum(abs(e.ercot_error_mw)) / sum(e.actual_mw),
           100 * sum(abs(l.forecast_mw - e.actual_mw)) / sum(e.actual_mw)
    FROM evaluation_dataset e
    JOIN latest l USING (weather_zone, target_ts_utc)
    WHERE l.rn = 1 AND e.ercot_forecast_mw IS NOT NULL
    """
).fetchone()

print(f"as-of, publication <= T-24h      WAPE {asof:5.2f}%")
print(f"latest vintage available today   WAPE {hindsight:5.2f}%")
print(f"                                 -> {asof / hindsight:.1f}x flattering")
print()
print("Nothing errored. No row went missing. One predicate changed.")

as-of, publication <= T-24h      WAPE  3.39%
latest vintage available today   WAPE  1.08%
                                 -> 3.1x flattering

Nothing errored. No row went missing. One predicate changed.


## What breaks

| Failure | How it presents | Detected by |
| --- | --- | --- |
| Cutoff collapsed to one global timestamp | Metrics improve. Nothing errors. | `CUTOFF_VIOLATION`, re-derived independently by the gate |
| Seasonal lag as `− INTERVAL 168 HOUR` | Wrong hour selected, twice a year | `tests/test_dst.py` |
| Actual selected against `T − 24h` | Most actuals vanish; the rest are fine | `MISSING_ACTUAL` |
| Naive input selected against `processing_ts` | **Leakage.** Model looks better. | Nothing automatic — this is why the clocks are documented |
| `INNER JOIN` instead of `LEFT JOIN` | Unservable hours vanish *from the denominator* | Row counts vs `targets` |
| Tie broken by `row_number() = 1` | One of two conflicting values silently wins | `*_value_count > 1` → `CONFLICTING_*` |

That fourth row has no automated detector, and that is worth saying out loud
rather than discovering in an interview.

## In production

**Materialize, do not view.** The output is a *historical* answer keyed by
processing date. A view re-computes against whatever the warehouse holds today,
which silently destroys the property the query exists to provide.

**Partition by target date** and prune. Here `forecast_vintage` is 12.6M rows for
one month; a year of retained vintages is well past 100M, and the eligibility
predicate is a full scan without pruning.

**Porting to another engine is where this breaks.** `TIMESTAMPTZ` semantics and
`AT TIME ZONE` direction differ between DuckDB, Snowflake, BigQuery and Spark —
in some engines `AT TIME ZONE` converts, in others it interprets. Port the DST
tests **first**, then the query.

**As a dbt model:** one model, `unique` test on `(weather_zone, target_ts_utc)`,
and a bespoke test asserting `ercot_publication_ts_utc <= cutoff_ts_utc` with
zero rows. That test is the whole contract. It should be impossible to merge
without it.

**Guard against well-meaning optimization.** The predicate references
`f.target_ts_utc` per row, which looks like something to hoist into a constant.
Hoisting it is the bug. A comment is not enough; the independent gate check and
`test_readiness_gate_does_not_trust_the_query_it_gates` are what hold the line.

## Runbook

| Symptom | Severity | Action |
| --- | --- | --- |
| `CUTOFF_VIOLATION > 0` | **Correctness incident.** Not a data-quality ticket. | Halt release. Quarantine every downstream artifact built from this run. Bisect the query change. |
| `MISSING_ASOF_FORECAST` on the first hour of the window | Expected | Structural if the publication window starts too late. Report; do not widen the window silently. |
| `CONFLICTING_ASOF_FORECAST` | Blocking | Two values at the same winning timestamp. Do **not** pick one. Ask the source which is authoritative. |
| `STALE_FORECAST_VINTAGE` | Investigate | Almost always retrieval, not ERCOT. See Playbook 0. |
| Row count ≠ zones × target hours | Blocking | A join changed grain. Compare against `targets`. |

## Where this lives

`sql/asof_join.sql` (authoritative) · `sql/asof_join_standalone.sql` (literals, for
a SQL client) · `tests/test_asof_join.py` · `tests/test_dst.py`